# MLflow Experiment Tracking Integration

**Purpose:** Track and compare all 4 models (Ridge + XGBoost for both challenges) with MLflow.

**Usage:** This notebook is designed to be **merged into DSC_Datathon.ipynb** or run standalone in Databricks.

---

## Setup Instructions

### In Databricks:
- MLflow is pre-installed and auto-configured
- Experiments are automatically saved to Databricks workspace

### Locally (optional):
```bash
pip install mlflow
```

---

## What This Adds

1. **Experiment creation** with descriptive name
2. **Automatic metric logging** (R², MAE, train/test split info)
3. **Parameter tracking** (model hyperparameters, feature counts)
4. **Artifact logging** (feature importance plots, model artifacts)
5. **Model comparison table** (side-by-side view of all 4 models)

In [ ]:
# ============================================================================
# CELL 0: MLflow Setup (ADD THIS AFTER YOUR IMPORTS IN DSC_Datathon.ipynb)
# ============================================================================

# Install MLflow if needed (Databricks has it pre-installed)
try:
    import mlflow
    import mlflow.sklearn
    print(f"MLflow version: {mlflow.__version__}")
except ImportError:
    print("Installing MLflow...")
    %pip install -q mlflow
    import mlflow
    import mlflow.sklearn
    print(f"MLflow version: {mlflow.__version__}")

# Create experiment (Databricks will use workspace; local uses ./mlruns)
EXPERIMENT_NAME = "DSC_Datathon_2026_Humanitarian_Analytics"

try:
    experiment_id = mlflow.create_experiment(
        EXPERIMENT_NAME,
        tags={
            "team": "dvislawa",
            "challenges": "geo_insight,beneficiary_targeting",
            "framework": "scikit-learn",
        }
    )
    print(f"Created experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
except mlflow.exceptions.MlflowException:
    # Experiment already exists
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

mlflow.set_experiment(EXPERIMENT_NAME)

print("\n✅ MLflow ready. All model runs will be tracked automatically.")

In [ ]:
# ============================================================================
# CELL 1: Helper Functions for MLflow Logging
# ============================================================================

import tempfile
import os
from datetime import datetime

def log_model_run(
    run_name: str,
    challenge: str,  # "geo_insight" or "beneficiary_targeting"
    model_type: str,  # "Ridge" or "XGBoost" or "GradientBoosting"
    pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    feature_names: list,
    num_features: list,
    cat_features: list,
    fig_importance=None,
    extra_params: dict = None,
    extra_metrics: dict = None,
):
    """
    Log a complete model run to MLflow with all relevant metadata.
    
    Returns: (run_id, metrics_dict)
    """
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
    
    with mlflow.start_run(run_name=run_name) as run:
        # ---- Tags ----
        mlflow.set_tags({
            "challenge": challenge,
            "model_type": model_type,
            "timestamp": datetime.now().isoformat(),
        })
        
        # ---- Parameters ----
        mlflow.log_param("n_train_samples", len(X_train))
        mlflow.log_param("n_test_samples", len(X_test))
        mlflow.log_param("n_features_total", len(feature_names))
        mlflow.log_param("n_numeric_features", len(num_features))
        mlflow.log_param("n_categorical_features", len(cat_features))
        mlflow.log_param("numeric_features", ",".join(num_features))
        mlflow.log_param("categorical_features", ",".join(cat_features))
        
        # Model-specific params
        model = pipeline.named_steps["model"]
        if hasattr(model, "alpha"):
            mlflow.log_param("alpha", model.alpha)
        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)
        if hasattr(model, "max_depth"):
            mlflow.log_param("max_depth", model.max_depth)
        if hasattr(model, "learning_rate"):
            mlflow.log_param("learning_rate", model.learning_rate)
        
        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, v)
        
        # ---- Metrics ----
        pred_train = pipeline.predict(X_train)
        pred_test = pipeline.predict(X_test)
        
        metrics = {
            "train_r2": r2_score(y_train, pred_train),
            "train_mae": mean_absolute_error(y_train, pred_train),
            "train_rmse": mean_squared_error(y_train, pred_train, squared=False),
            "test_r2": r2_score(y_test, pred_test),
            "test_mae": mean_absolute_error(y_test, pred_test),
            "test_rmse": mean_squared_error(y_test, pred_test, squared=False),
        }
        
        if extra_metrics:
            metrics.update(extra_metrics)
        
        mlflow.log_metrics(metrics)
        
        # ---- Artifacts ----
        # Log feature importance plot if provided
        if fig_importance is not None:
            with tempfile.TemporaryDirectory() as tmpdir:
                fig_path = os.path.join(tmpdir, "feature_importance.png")
                fig_importance.savefig(fig_path, dpi=150, bbox_inches="tight")
                mlflow.log_artifact(fig_path, "plots")
        
        # Log the model
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"\n✅ Logged run: {run_name}")
        print(f"   Run ID: {run.info.run_id}")
        print(f"   Test R²: {metrics['test_r2']:.4f} | Test MAE: {metrics['test_mae']:.4f}")
        
        return run.info.run_id, metrics


def get_comparison_table(experiment_name: str = EXPERIMENT_NAME):
    """
    Get a comparison DataFrame of all runs in the experiment.
    """
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        print(f"No experiment found: {experiment_name}")
        return None
    
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.test_r2 DESC"],
    )
    
    if runs_df.empty:
        print("No runs found in experiment.")
        return None
    
    # Select key columns for comparison
    key_cols = [
        "run_id",
        "tags.challenge",
        "tags.model_type",
        "params.n_train_samples",
        "params.n_test_samples",
        "metrics.train_r2",
        "metrics.test_r2",
        "metrics.train_mae",
        "metrics.test_mae",
    ]
    
    available_cols = [c for c in key_cols if c in runs_df.columns]
    comparison = runs_df[available_cols].copy()
    
    # Clean column names
    comparison.columns = [
        c.replace("tags.", "").replace("params.", "").replace("metrics.", "")
        for c in comparison.columns
    ]
    
    return comparison


print("✅ MLflow helper functions loaded.")

---

## Integration Points

Below are the **exact code snippets to add to your existing model cells** in `DSC_Datathon.ipynb`.

---

In [ ]:
# ============================================================================
# CELL 19 REPLACEMENT: Geo-Insight Ridge + XGBoost with MLflow
# ============================================================================
# Replace or augment your existing Cell 19 with this version.
# Prerequisites: core_enriched DataFrame must exist from earlier cells.

# --- Standard imports (already in your notebook) ---
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- MLflow imports ---
import mlflow
import mlflow.sklearn

# Ensure experiment is set
mlflow.set_experiment("DSC_Datathon_2026_Humanitarian_Analytics")

# --- Data prep (same as your original) ---
model_df = core_enriched.dropna(subset=["log10_usd_per_in_need"]).copy()
model_df["crisis_type_primary"] = model_df["crisis_type"].astype(str).str.split("|").str[0].str.strip()
model_df["driver_primary"] = model_df["primary_driver"].astype(str).str.strip()

train_df = model_df[model_df["year"].isin([2024, 2025])].copy()
test_df = model_df[model_df["year"] == 2026].copy()

num_features = [
    "need_rate",
    "log10_in_need",
    "severity_index",
    "complexity",
    "operating_env",
    "years_since_first_response",
]
cat_features = [
    "region",
    "crisis_type_primary",
    "driver_primary",
]

num_features = [c for c in num_features if c in model_df.columns]
cat_features = [c for c in cat_features if c in model_df.columns]

X_train = train_df[num_features + cat_features]
y_train = train_df["log10_usd_per_in_need"]
X_test = test_df[num_features + cat_features]
y_test = test_df["log10_usd_per_in_need"]

# --- Preprocessing ---
pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
            cat_features,
        ),
    ]
)

# ==========================================================================
# MODEL 1: Ridge Regression (Geo-Insight) with MLflow
# ==========================================================================

pipe_ridge = Pipeline(
    steps=[
        ("pre", pre),
        ("model", Ridge(alpha=1.0)),
    ]
)

pipe_ridge.fit(X_train, y_train)

# Extract feature names
feature_names_ridge = []
feature_names_ridge.extend(num_features)
if cat_features:
    ohe = pipe_ridge.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_ridge.extend(ohe.get_feature_names_out(cat_features).tolist())

# Create feature importance plot
coefs = pipe_ridge.named_steps["model"].coef_
coef_df = (
    pd.DataFrame({"feature": feature_names_ridge, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

plot_coef = coef_df.head(18).sort_values("coef")
fig_ridge, ax = plt.subplots(figsize=(10, 7))
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Ridge: Factors associated with $/person-in-need (Geo-Insight)")
ax.set_xlabel("Ridge coefficient (standardized)")
plt.tight_layout()

# Log to MLflow
run_id_ridge, metrics_ridge = log_model_run(
    run_name="GeoInsight_Ridge_v1",
    challenge="geo_insight",
    model_type="Ridge",
    pipeline=pipe_ridge,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=feature_names_ridge,
    num_features=num_features,
    cat_features=cat_features,
    fig_importance=fig_ridge,
    extra_params={
        "train_years": "2024,2025",
        "test_year": "2026",
        "target": "log10_usd_per_in_need",
    },
)

plt.show()

# ==========================================================================
# MODEL 2: XGBoost/GradientBoosting (Geo-Insight) with MLflow
# ==========================================================================

try:
    from xgboost import XGBRegressor
    tree_model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    tree_model_name = "XGBoost"
    tree_ohe_sparse = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    tree_model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    tree_model_name = "GradientBoosting"
    tree_ohe_sparse = False

try:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

pre_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
            cat_features,
        ),
    ]
)

pipe_tree = Pipeline(steps=[("pre", pre_tree), ("model", tree_model)])
pipe_tree.fit(X_train, y_train)

# Extract feature names
feature_names_tree = []
feature_names_tree.extend(num_features)
if cat_features:
    ohe_tree = pipe_tree.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_tree.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

# Create feature importance plot
importances = pipe_tree.named_steps["model"].feature_importances_
imp_df = (
    pd.DataFrame({"feature": feature_names_tree, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(18)
    .sort_values("importance")
)

fig_tree, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
ax.set_title(f"{tree_model_name}: Feature importance (Geo-Insight)")
ax.set_xlabel("Feature importance")
plt.tight_layout()

# Log to MLflow
run_id_tree, metrics_tree = log_model_run(
    run_name=f"GeoInsight_{tree_model_name}_v1",
    challenge="geo_insight",
    model_type=tree_model_name,
    pipeline=pipe_tree,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=feature_names_tree,
    num_features=num_features,
    cat_features=cat_features,
    fig_importance=fig_tree,
    extra_params={
        "train_years": "2024,2025",
        "test_year": "2026",
        "target": "log10_usd_per_in_need",
    },
)

plt.show()

print("\n" + "="*60)
print("GEO-INSIGHT MODEL COMPARISON")
print("="*60)
print(f"Ridge:         Test R² = {metrics_ridge['test_r2']:.4f} | MAE = {metrics_ridge['test_mae']:.4f}")
print(f"{tree_model_name}: Test R² = {metrics_tree['test_r2']:.4f} | MAE = {metrics_tree['test_mae']:.4f}")
print("="*60)

In [ ]:
# ============================================================================
# CELL 30 REPLACEMENT: Challenge 1 Ridge + XGBoost with MLflow
# ============================================================================
# Replace or augment your existing Cell 30 with this version.
# Prerequisites: model_df DataFrame must exist from Cell 29.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import mlflow
import mlflow.sklearn

mlflow.set_experiment("DSC_Datathon_2026_Humanitarian_Analytics")

MIN_BEN_REVIEW = 50

feat_df = model_df.dropna(subset=["budget_usd", "beneficiaries_total", "log10_cpb"]).copy()
feat_df_eff = feat_df[feat_df["beneficiaries_total"] >= MIN_BEN_REVIEW].copy()

# We'll run on the filtered dataset (beneficiaries >= 50) for cleaner signal
df = feat_df_eff.copy()
df["log10_budget"] = np.log10(df["budget_usd"].where(df["budget_usd"] > 0))
df["log10_beneficiaries"] = np.log10(df["beneficiaries_total"].where(df["beneficiaries_total"] > 0))

target = df["log10_cpb"]

num_features = [
    "log10_budget",
    "log10_beneficiaries",
    "support_cost_share",
    "duration_months",
    "beneficiary_share_of_country_pop",
]
cat_features = [
    "cluster_primary",
    "OrganizationType",
    "AllocationTypeCategory",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

X = df[num_features + cat_features]
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.25, random_state=42)

# --- Preprocessing ---
pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
            cat_features,
        ),
    ]
)

# ==========================================================================
# MODEL 3: Ridge Regression (Challenge 1 - CPB) with MLflow
# ==========================================================================

pipe_ridge_cpb = Pipeline(
    steps=[
        ("pre", pre),
        ("model", Ridge(alpha=1.0)),
    ]
)

pipe_ridge_cpb.fit(X_train, y_train)

# Extract feature names
feature_names_cpb = []
feature_names_cpb.extend(num_features)
if cat_features:
    ohe = pipe_ridge_cpb.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_cpb.extend(ohe.get_feature_names_out(cat_features).tolist())

# Create feature importance plot
coefs = pipe_ridge_cpb.named_steps["model"].coef_
coef_df = (
    pd.DataFrame({"feature": feature_names_cpb, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

plot_coef = coef_df.head(18).sort_values("coef")
fig_ridge_cpb, ax = plt.subplots(figsize=(10, 7))
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Ridge: Factors associated with log10(CPB) (Challenge 1)")
ax.set_xlabel("Ridge coefficient (standardized)")
plt.tight_layout()

# Log to MLflow
run_id_ridge_cpb, metrics_ridge_cpb = log_model_run(
    run_name="Challenge1_Ridge_CPB_v1",
    challenge="beneficiary_targeting",
    model_type="Ridge",
    pipeline=pipe_ridge_cpb,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=feature_names_cpb,
    num_features=num_features,
    cat_features=cat_features,
    fig_importance=fig_ridge_cpb,
    extra_params={
        "min_beneficiaries": MIN_BEN_REVIEW,
        "target": "log10_cpb",
        "split": "random_75_25",
    },
)

plt.show()

# ==========================================================================
# MODEL 4: XGBoost/GradientBoosting (Challenge 1 - CPB) with MLflow
# ==========================================================================

try:
    from xgboost import XGBRegressor
    tree_model_cpb = XGBRegressor(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    tree_model_name_cpb = "XGBoost"
    tree_ohe_sparse = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    tree_model_cpb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    tree_model_name_cpb = "GradientBoosting"
    tree_ohe_sparse = False

try:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

pre_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
            cat_features,
        ),
    ]
)

pipe_tree_cpb = Pipeline(steps=[("pre", pre_tree), ("model", tree_model_cpb)])
pipe_tree_cpb.fit(X_train, y_train)

# Extract feature names
feature_names_tree_cpb = []
feature_names_tree_cpb.extend(num_features)
if cat_features:
    ohe_tree = pipe_tree_cpb.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_tree_cpb.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

# Create feature importance plot
importances = pipe_tree_cpb.named_steps["model"].feature_importances_
imp_df = (
    pd.DataFrame({"feature": feature_names_tree_cpb, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(12)
    .sort_values("importance")
)

fig_tree_cpb, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
ax.set_title(f"{tree_model_name_cpb}: Feature importance (Challenge 1 - CPB)")
ax.set_xlabel("Feature importance")
plt.tight_layout()

# Log to MLflow
run_id_tree_cpb, metrics_tree_cpb = log_model_run(
    run_name=f"Challenge1_{tree_model_name_cpb}_CPB_v1",
    challenge="beneficiary_targeting",
    model_type=tree_model_name_cpb,
    pipeline=pipe_tree_cpb,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=feature_names_tree_cpb,
    num_features=num_features,
    cat_features=cat_features,
    fig_importance=fig_tree_cpb,
    extra_params={
        "min_beneficiaries": MIN_BEN_REVIEW,
        "target": "log10_cpb",
        "split": "random_75_25",
    },
)

plt.show()

print("\n" + "="*60)
print("CHALLENGE 1 (CPB) MODEL COMPARISON")
print("="*60)
print(f"Ridge:         Test R² = {metrics_ridge_cpb['test_r2']:.4f} | MAE = {metrics_ridge_cpb['test_mae']:.4f}")
print(f"{tree_model_name_cpb}: Test R² = {metrics_tree_cpb['test_r2']:.4f} | MAE = {metrics_tree_cpb['test_mae']:.4f}")
print("="*60)

In [ ]:
# ============================================================================
# FINAL CELL: Model Comparison Summary (ADD AT END OF NOTEBOOK)
# ============================================================================

print("\n" + "="*80)
print("           DSC DATATHON 2026 - COMPLETE MODEL COMPARISON")
print("="*80)

# Get comparison table from MLflow
comparison = get_comparison_table()

if comparison is not None and not comparison.empty:
    print("\nAll tracked experiments:")
    print(comparison.to_string(index=False))
    
    # Summary by challenge
    print("\n" + "-"*80)
    print("BEST MODEL BY CHALLENGE (highest test R²):")
    print("-"*80)
    
    if "challenge" in comparison.columns and "test_r2" in comparison.columns:
        best_per_challenge = comparison.loc[comparison.groupby("challenge")["test_r2"].idxmax()]
        for _, row in best_per_challenge.iterrows():
            print(f"  {row['challenge']:25} -> {row['model_type']:15} (R² = {row['test_r2']:.4f})")
else:
    print("\nNo MLflow runs found. Run the model cells above first.")

print("\n" + "="*80)
print("To view detailed results in Databricks:")
print("  1. Click 'Experiments' in left sidebar")
print("  2. Select 'DSC_Datathon_2026_Humanitarian_Analytics'")
print("  3. Compare runs side-by-side")
print("="*80)

---

## Quick Integration Guide

### Option A: Merge into DSC_Datathon.ipynb

1. **Copy Cell 0** (MLflow setup) → Insert **after Cell 1** (imports)
2. **Copy Cell 1** (helper functions) → Insert **after new Cell 0**
3. **Replace Cell 19** with the Geo-Insight MLflow version
4. **Replace Cell 30** with the Challenge 1 MLflow version
5. **Add the final comparison cell** at the end of the notebook

### Option B: Run This Notebook Standalone

1. Open this notebook in Databricks
2. Run Cells 0-1 (setup)
3. Ensure `core_enriched` and `model_df` DataFrames exist (from running the data loading cells of DSC_Datathon.ipynb)
4. Run the model cells
5. View results in MLflow UI

---

## Video Presentation Tips

With MLflow tracking, you can show:

1. **MLflow Experiments UI** - Side-by-side model comparison
2. **Metric trends** - If you run multiple versions
3. **Artifact browser** - Show feature importance plots
4. **Professional workflow** - "We used MLflow to systematically compare 4 models..."

Screenshot the MLflow comparison table for your video.